# Colab launcher

Runs the GPU work on Colab. This notebook holds no project logic: it prepares the VM and calls the
same commands as on a local machine, with `GEO_OVERLAYS=configs/colab.yaml` so every command uses
the Colab paths (data and caches on the VM's local disk, run outputs on Drive).

**Runtime → Change runtime type:** A100 for full fine-tunes; L4 is enough for the smoke runs and the
probe. Avoid T4 (no bfloat16) and TPUs.

**Colab secrets (key icon), notebook access on:** `GH_TOKEN` (GitHub, read, while the repo is
private) and `HF_TOKEN` (Hugging Face; DINOv3 access must be granted on the hub).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/geo-relations"
!mkdir -p {DRIVE}/runs {DRIVE}/data_tarballs

In [ ]:
import os
from google.colab import userdata

REPO = "parsa-hke/veridiagram"  # GitHub owner/name
BRANCH = "main"
WORKDIR = "/content/veridiagram"

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

token = secret("GH_TOKEN")
url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
if os.path.isdir(f"{WORKDIR}/.git"):
    !git -C {WORKDIR} pull --ff-only
else:
    !git clone -q -b {BRANCH} {url} {WORKDIR}
    !git -C {WORKDIR} remote set-url origin https://github.com/{REPO}.git  # drop token from .git/config
%cd {WORKDIR}
!git log -1 --oneline

In [ ]:
# Pinned versions (numpy/pandas match Colab's own packages). If pip asks for a restart, use
# Runtime -> Restart session, rerun the cells above (idempotent) and continue.
!pip install -q -r requirements.txt

In [ ]:
os.environ["HF_HOME"] = "/content/hf_cache"  # local disk: safetensors are memory-mapped, which the Drive mount does not support reliably
os.environ["GEO_OVERLAYS"] = "configs/colab.yaml"       # every CLI uses the Colab paths
if secret("HF_TOKEN"):
    os.environ["HF_TOKEN"] = secret("HF_TOKEN")
!python -m experiments.run --device-info

## Data

First session: downloads the datasets, builds everything (`python -m data.prepare`, ~15-20 min) and
saves one tarball to Drive. Later sessions: unpack that tarball to the local disk (~2 min). The
tarball name carries the git hash of the data pipeline, so a pipeline change rebuilds automatically.

In [ ]:
# The tarball is named after data/VERSION (bumped only when the data pipeline's outputs change).
VERSION = open("data/VERSION").read().split()[0]
tarball = f"{DRIVE}/data_tarballs/data_v{VERSION}.tar"
legacy = f"{DRIVE}/data_tarballs/data_4fa8896.tar"          # first tarball, identical to v1
if not os.path.exists(tarball) and VERSION == "1" and os.path.exists(legacy):
    os.rename(legacy, tarball)
print("data version", VERSION, "->", tarball)
if os.path.exists(tarball):
    !mkdir -p /content/data && tar -C /content -xf {tarball}
else:
    !python -m data.prepare --download
    !tar -C /content -cf {tarball} data
!du -sh /content/data/raw /content/data/processed

## Phase 6 gate: smoke runs (must pass before any full run)

In [ ]:
# DINOv3 access check: loads the gated checkpoint with NO fallback; must end with "OK".
!python -m models.backbone

In [ ]:
!python -m experiments.run --config configs/probe.yaml --smoke
!python -m experiments.run --config configs/finetune.yaml --smoke
!python -m experiments.gate smoke

## Lab checkpoints

The lab runs that the last Colab batch needs, unpacked once from the lab archives on Drive
(`geo-relations/lab_results/`) into `runs/` and checked against their sha256 (computed on the laptop):
the three 7.3 checkpoints (`finetune_synth10k_s2` is the Part B teacher) from `geo_ckpt_synth10k.tar`,
the three probe heads and the six run configs from `geo_results_20261012.tar.gz` (the square-padded
re-evaluation compares each source run's saved config with its own).

In [ ]:
import hashlib, tarfile
LAB = f"{DRIVE}/lab_results"
LAB_FILES = {   # archive -> {member: sha256 verified on the laptop}
    "geo_ckpt_synth10k.tar": {
        "runs/finetune_synth10k_s0/checkpoints/best.pt": "e17458b6233e8a6d71361fe8facc5a7a8168946846cd708fdbdbaa267beef599",
        "runs/finetune_synth10k_s1/checkpoints/best.pt": "3b3faedec35fc2a6bb973d219701fd0126d361275c0ae62e70559e89512a4bd5",
        "runs/finetune_synth10k_s2/checkpoints/best.pt": "07db988c20212b7275ab38316d22890271c4fb6c76070aa5a98d1a1d994d0bb9",  # teacher
    },
    "geo_results_20261012.tar.gz": {
        "runs/probe_s0/probe_head.pt": "b2bf2ef03bc2f7125290bab5504afe2ecc12677d9dc9b92891aa5137cee96143",
        "runs/probe_s1/probe_head.pt": "80175066f2cd9252f653ad8f8d747c4ba8a0fb6f629436b1defaf4e72f6227b0",
        "runs/probe_s2/probe_head.pt": "a8c529f4ea0f314499caa98253da202718e466398e719f34c5daef886c586915",
        "runs/probe_s0/config.yaml": "434b083f1b236902ec124b136e918c79bb0f850c716662b6e1ed3e788e3f08f4",
        "runs/probe_s1/config.yaml": "1b34f4c2f94a033856a56a66467179e88110c1a165d5b5ec0b2317808734f133",
        "runs/probe_s2/config.yaml": "5ee271cbbc5a0919ef7ee488ce017d4e4014ca439693ed15b073544b194e157a",
        "runs/finetune_synth10k_s0/config.yaml": "4f2542bddaaeedbe860410b402b756b726270325c104222b0dab1e51fe242a67",
        "runs/finetune_synth10k_s1/config.yaml": "cfed7846a9b193aa11c5a6cace1f19e48b06e6e4badc5b030fcde19d3d768c85",
        "runs/finetune_synth10k_s2/config.yaml": "09f3ccaceabba92b1952a13bbef13ad60eabbc74ec4256e9afe82dd799d669a0",
    },
}

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 24), b""):
            h.update(block)
    return h.hexdigest()

for archive, members in LAB_FILES.items():
    for member, digest in members.items():
        dest = f"{DRIVE}/{member}"
        if not os.path.exists(dest) or sha256(dest) != digest:
            with tarfile.open(f"{LAB}/{archive}") as t:
                t.extract(member, path=DRIVE, filter="data")
        ok = sha256(dest) == digest
        print(f"{member:48s} {os.path.getsize(dest):>10d} bytes, sha256 {'OK' if ok else 'MISMATCH'}")
        assert ok, f"{member} differs from the lab copy: delete {dest} and rerun this cell"

## Phase 7 gates: smoke runs of the new modes (must pass before the queue)

The no-overlay ablation (7.8) and self-training (Part B, with the real teacher) on the smoke subset,
then the square-padded re-evaluation (plan §0.2 #10 note) of the Phase 6 smoke fine-tune and probe
above, then the two gate checks. About 8 minutes.

In [ ]:
!python -m experiments.run --config configs/ablation_nooverlay.yaml --smoke
!python -m experiments.run --config configs/selftrain_consistency.yaml --smoke
!python -m experiments.gate phase7
!python -m experiments.run --config configs/eval_square.yaml --smoke --set name=sq_finetune --set eval.source_config=configs/finetune.yaml
!python -m experiments.run --config configs/eval_square.yaml --smoke --set name=sq_probe --set eval.source_config=configs/probe.yaml
!python -m experiments.gate eval_square

## Full runs

`configs/queue_phase7_final.yaml` (29 runs, one after another): the Part B control seeds 1 and 2,
the square-padded re-evaluation of every Phase 7 checkpoint (`sq_<experiment>_s<seed>`), then 7.2
retrained (`finetune_colab`, the lab did not keep its checkpoints) and re-evaluated. Results go to
`MyDrive/geo-relations/runs/<name>_s<seed>/` and `runs/log.csv`. If the session disconnects, run every
cell again from the top: finished runs are skipped and an interrupted fine-tune resumes from its last
epoch.

In [ ]:
RUN_FULL = False   # set to True only after all three gates above printed PASS ("Run all" never starts full runs)
QUEUE = "configs/queue_phase7_final.yaml"
!python -m experiments.queue {QUEUE} --status
if RUN_FULL:
    !python -m experiments.queue {QUEUE}
else:
    print("RUN_FULL is False: skipping", QUEUE)

## Pack the results for the laptop

Metrics, predictions and histories of every run on Drive (no checkpoints) into one small file:
`geo-relations/geo_results_colab_final.tar.gz`. Download it to the laptop when the queue has finished.

In [ ]:
!python -m experiments.queue {QUEUE} --status
!cd {DRIVE} && tar --exclude='*/checkpoints' --exclude='selftrain_teacher/*_smoke' -czf geo_results_colab_final.tar.gz runs && ls -lh geo_results_colab_final.tar.gz